# Messages in Langchain

In LangChain, a **message** is a single turn in a conversation. Chat models don't
take plain strings internally — they work with a **list of messages**, each
tagged with a **role** (who is speaking). This is how the model knows the
difference between instructions, user input, and its own replies.

### Why messages?

A conversation is an ordered list of messages:

```text
[ System  ] → the rules/persona
[ Human   ] → the user's question
[ AI      ] → the model's reply
[ Tool    ] → a tool's result
[ Human   ] → follow-up question
...
```

Each new turn is **appended** to the list, so the model always sees the full
history.

### The main message types

| Message | Role | Purpose |
|---------|------|---------|
| `SystemMessage` | system | Sets behavior/persona/rules (usually first) |
| `HumanMessage` | user | The user's input |
| `AIMessage` | assistant | The model's response (may contain tool calls) |
| `ToolMessage` | tool | The result of running a tool |

### 1. Importing messages

```python
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)
```

### 2. SystemMessage — set the behavior

- Defines how the model should act. Usually placed **first** in the list.
- Tells LLM how it will behave and provide **context** for conversations.

```python
SystemMessage(content="You are a helpful assistant that answers in simple English.")
```

### 3. HumanMessage — the user's input

- Represents User input and interaction with model

```python
HumanMessage(content="What is LangChain?")
```

### 4. AIMessage — the model's reply

- Returned by the model. Its `.content` holds the text; `.tool_calls` holds any tool requests.
- Response generated by the model, including text content, tool calss, and meta data

```python
AIMessage(content="LangChain is a framework for building LLM apps.")
```

### 5. ToolMessage — a tool's result

Sent **back** to the model after you run a tool. It links to the request via
`tool_call_id`.

```python
ToolMessage(content="-6", tool_call_id="call_123")
```

### 6. Building a conversation

```python
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is 2 plus 8?"),
]

response = model.invoke(messages)   # returns an AIMessage
print(response.content)

messages.append(response)           # keep the history growing
```

### 7. Dictionary shorthand

Instead of message classes, you can use simple dicts with `role` + `content`.
LangChain converts them automatically.

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What is 2 plus 8?"},
]

response = model.invoke(messages)
```

| Role string | Equivalent class |
|-------------|------------------|
| `"system"` | `SystemMessage` |
| `"user"` | `HumanMessage` |
| `"assistant"` | `AIMessage` |
| `"tool"` | `ToolMessage` |

### 8. Useful attributes on a message

```python
response = model.invoke("Hello")

print(response.content)      # the text
print(response.type)         # 'ai', 'human', 'system', 'tool'
print(response.tool_calls)   # tool requests (for AIMessage)
```

### How messages flow in a tool loop

```text
SystemMessage   → rules
HumanMessage    → "What is 2 minus 8?"
AIMessage       → tool_call: subtract_numbers(2, 8)
ToolMessage     → "-6"   (result you append after running the tool)
AIMessage       → "The result of 2 minus 8 is -6."
```

Every step **adds a message** to the list, so the model always has full context.

### Key notes

- A conversation = an **ordered list of messages**.
- **Always append** new replies to keep history — the model has no memory on its
  own between calls.
- `SystemMessage` usually goes **first** and sets the tone.
- `AIMessage.tool_calls` holds tool requests; `ToolMessage` returns the results.
- **Dicts (`role`/`content`) and message classes are interchangeable** — use
  whichever is cleaner.

In [2]:
import os, httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

http_client = httpx.Client(verify=False)          # ⚠️ dev only

model = ChatOpenAI(                                # 👈 REBUILD model here
    model="llama3.2",
    base_url=os.getenv("GE_BASE_URL"),
    api_key=os.getenv("GE_API_KEY"),
    temperature=0,
    http_client=http_client,                       # 👈 attach the client
)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000025232A96720>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000025233511040>, root_client=<openai.OpenAI object at 0x0000025232A95040>, root_async_client=<openai.AsyncOpenAI object at 0x0000025233499040>, model_name='llama3.2', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://genai-imaging-lab.apps.ge-healthcare.net:4000/v1', openai_proxy=None, http_client=<httpx.Client object at 0x00000252329D0DA0>, stream_chunk_timeout=120.0)

In [3]:
model.invoke("What is AI?")

AIMessage(content='AI, or Artificial Intelligence, refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:\n\n1. Learning: The ability to improve performance on a task over time, without being explicitly programmed.\n2. Problem-solving: The ability to analyze problems and find solutions.\n3. Reasoning: The ability to draw conclusions based on available information.\n4. Perception: The ability to interpret and understand data from sensors or other sources.\n\nThere are several types of AI, including:\n\n1. Narrow or Weak AI: Designed to perform a specific task, such as facial recognition or language translation.\n2. General or Strong AI: A hypothetical AI system that possesses the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence.\n3. Superintelligence: An AI system that significantly surpasses human intelligence in a broad range of cognitive tasks.\n\nAI can be c

## Text Prompt (`PromptTemplate`)

### What it is
A reusable template for a **single string** with `{placeholders}` filled at runtime.

### When to use
- Simple, single-string prompts (text in → text out).
- Reusing one prompt with different values.
- You want minmal code complexity
- You don't need the conversation history



In [4]:
model.invoke("What is my name? Explain in 20 words.")

AIMessage(content="I don't have information about your identity, so I'm unable to provide your name. You're a unique individual.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 36, 'total_tokens': 61, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'ollama_chat/llama3.2', 'system_fingerprint': None, 'id': 'chatcmpl-7b6a324e-8e0e-4c99-995b-45b304c0ced0', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fb70d-973a-7523-abb4-c3935676c3f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 25, 'total_tokens': 61, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are expert in coding for python language."),
    # HumanMessage("Write a function to integrate a model with langchain."), -> Model is outDATED IT IS GENERATING RESPINSE BASED ON PREVIOUS INFO
    HumanMessage("Write a function for checking Prime Numbers."),
]

In [6]:
response = model.invoke(messages)

In [7]:
print(response.content)

**Prime Number Checker Function**

Here's an example of a Python function that checks if a number is prime:

```python
def is_prime(n):
    """
    Checks if a number is prime.

    Args:
        n (int): The number to check.

    Returns:
        bool: True if the number is prime, False otherwise.
    """

    # Handle edge cases
    if n <= 1:
        return False

    # Check divisibility from 2 to sqrt(n)
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False

    # If no divisors found, the number is prime
    return True
```

**Example Use Cases**
--------------------

```python
# Test with a known prime number
print(is_prime(25))  # Output: False

# Test with a known non-prime number
print(is_prime(30))  # Output: False

# Test with a small prime number
print(is_prime(7))   # Output: True

# Test with a large prime number
print(is_prime(997)) # Output: True
```

**Explanation**
---------------

This function uses a simple trial division method 

In [8]:
sys_msg = SystemMessage("You are a helpful coding assistant.")
messages2 =[
    sys_msg,
    HumanMessage("How do I create a Rest API? Provide links to resources and examples."),
]
response2 = model.invoke(messages2)
print(response2.content)

Creating a REST (Representational State of Resource) API involves several steps, including designing the API endpoints, defining the data models, implementing the API using a programming language, and testing it. Here's a step-by-step guide on how to create a REST API:

**Step 1: Design the API Endpoints**

* Identify the resources that will be exposed by your API (e.g., users, products, orders).
* Determine the actions that can be performed on these resources (e.g., create, read, update, delete).
* Use HTTP methods to define the behavior of each endpoint:
	+ GET: Retrieve a resource or a list of resources.
	+ POST: Create a new resource.
	+ PUT/PATCH: Update an existing resource.
	+ DELETE: Delete a resource.

**Step 2: Define the Data Models**

* Identify the data structures that will be used to represent each resource (e.g., JSON objects, database tables).
* Determine the fields that should be included in each data structure.
* Use a schema definition language like JSON Schema or Op

In [9]:
# Detail information as SystemMessage
sys_msg_2 = SystemMessage("""
You are a senior typescript developer with expertise in web development.
Always provide a code examples and explain your reasoning.
Be concise but thorough in your explanations. Avoid unnecessary verbosity.
""")

messages3 = [
    sys_msg_2,
    HumanMessage("Write a function to create a REST API in TypeScript."),
]

response3 = model.invoke(messages3)
print(response3.content)

**Creating a REST API using TypeScript**

In this example, we will create a simple REST API using TypeScript that allows us to manage books.

### Dependencies

We will use the following dependencies:

* `express`: A popular Node.js web framework for building web applications.
* `typescript`: The TypeScript compiler for JavaScript.

### Installation

To install the required dependencies, run the following command in your terminal:
```bash
npm init -y
npm install express typescript @types/express
```
### Project Structure

Create a new file called `app.ts` and add the following code:
```typescript
import express from 'express';

const app = express();

// Middleware to parse JSON bodies
app.use(express.json());

// In-memory data store for books
let books: { title: string; author: string }[] = [
  { title: 'Book 1', author: 'Author 1' },
  { title: 'Book 2', author: 'Author 2' },
];

// GET /books
app.get('/books', (req, res) => {
  res.json(books);
});

// POST /books
app.post('/books',

In [10]:

human_msg = HumanMessage(
    content=(
        "Write a TypeScript function that captures runtime JavaScript errors in the "
        "browser. Requirements:\n"
        "1. Use window.addEventListener('error', ...) to catch runtime errors.\n"
        "2. Use window.addEventListener('unhandledrejection', ...) for promise rejections.\n"
        "3. For each captured error, log the message, source file, line, and column.\n"
        "4. Store captured errors in an array and return a function to retrieve them.\n"
        "Do NOT assume console.error throws exceptions. "
        "Include a short usage example."
    ),
    name="developer",          # optional: identify different users
    id="devtools-checker1",    # unique identifier for tracing
)

In [11]:
response4 = model.invoke([sys_msg_2, human_msg])
print(response4.content)

**Error Capture Function**
```typescript
interface Error {
  message: string;
  filename: string;
  lineno: number;
  colno: number;
}

function captureErrors(): (error: Error) => void {
  const errors: Error[] = [];

  window.addEventListener('error', (event: ErrorEvent) => {
    if (event.error instanceof TypeError || event.error instanceof RangeError) {
      const error: Error = {
        message: event.error.message,
        filename: event.error.filename,
        lineno: event.error.lineno,
        colno: event.error.colno,
      };
      errors.push(error);
    }
  });

  window.addEventListener('unhandledrejection', (event: UnhandledRejectionEvent) => {
    const error: Error = {
      message: event.reason.message,
      filename: event.reason.filename,
      lineno: event.reason.lineno,
      colno: event.reason.colno,
    };
    errors.push(error);
  });

  return () => {
    console.log('Captured Errors:');
    for (const error of errors) {
      console.error(`Error Messag

In [14]:
response4.usage_metadata

{'input_tokens': 155,
 'output_tokens': 519,
 'total_tokens': 674,
 'input_token_details': {},
 'output_token_details': {}}

#### Messages can be passed to the model for context

In [12]:
#NOTE  Entire list of messages can be passed to the model for context, and the model will respond accordingly.

ai_msg = AIMessage("I'd happy to help you with that question!.")

messages = [
    SystemMessage("You are a helpful assistant that can perform mathematical operations and provide explanations."),
    HumanMessage("Can you help me subtract two numbers?"),
    ai_msg,
    HumanMessage("Please subtract 10 from 25."),
]

response = model.invoke(messages)
print(response.content)


To subtract 10 from 25, I'll perform the calculation:

25 - 10 = 15

The result is 15. Let me know if you have any other questions or need further assistance!


In [13]:
response.usage_metadata

{'input_tokens': 75,
 'output_tokens': 43,
 'total_tokens': 118,
 'input_token_details': {},
 'output_token_details': {}}

#### Tool messages

In [15]:
from langchain.messages import ToolMessage

In [16]:
# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [17]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [19]:
print(response.content)

I'm not able to provide real-time weather information. However, I can suggest some ways for you to find out the current weather in San Francisco:

1. Check online weather websites: You can check websites like weather.com, accuweather.com, or wunderground.com for the current weather conditions in San Francisco.
2. Use a mobile app: You can download mobile apps like Dark Sky or Weather Underground that provide real-time weather information for various locations, including San Francisco.
3. Tune into local news: You can watch local news channels or listen to radio stations that provide weather updates for San Francisco.
4. Check social media: Many weather services and meteorologists share weather updates on their social media accounts, so you can check those for the latest information.

Please note that the weather in San Francisco can change rapidly, especially during the summer months when fog rolls in suddenly. It's always a good idea to check the weather forecast before heading out.


In [20]:
response.usage_metadata

{'input_tokens': 50,
 'output_tokens': 192,
 'total_tokens': 242,
 'input_token_details': {},
 'output_token_details': {}}

In [29]:
import os, httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

http_client = httpx.Client(verify=False)          # ⚠️ dev only

model2 = ChatOpenAI(                                # 👈 REBUILD model here
    model="gemma3:27b",
    base_url=os.getenv("GE_BASE_URL"),
    api_key=os.getenv("GE_API_KEY"),
    temperature=0,
    http_client=http_client,                       # 👈 attach the client
)
model2

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x00000252335FC4A0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000252335DD6D0>, root_client=<openai.OpenAI object at 0x00000252335BB7D0>, root_async_client=<openai.AsyncOpenAI object at 0x00000252335B9430>, model_name='gemma3:27b', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://genai-imaging-lab.apps.ge-healthcare.net:4000/v1', openai_proxy=None, http_client=<httpx.Client object at 0x000002522F8C9970>, stream_chunk_timeout=120.0)

In [27]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages2 = [
    SystemMessage("You are expert in coding for python language."),
    HumanMessage("Write a function to integrate a model with langchain."), 
]

In [30]:
response = model2.invoke(messages2)

In [32]:
print(response.content)

```python
from langchain.chains import LLMChain
from langchain.llms import OpenAI  # Or any other LLM you prefer
from langchain.prompts import PromptTemplate

def integrate_model_with_langchain(model_name="gpt-3.5-turbo", api_key=None, prompt_template=None, input_text=None):
    """
    Integrates a language model (e.g., OpenAI) with Langchain for text generation.

    Args:
        model_name (str, optional): The name of the language model to use. Defaults to "gpt-3.5-turbo".
        api_key (str, optional): Your API key for the language model.  Required for most models.
        prompt_template (str, optional): A string containing the prompt template.  
                                         Use curly braces {} for variables.  
                                         Example: "Translate the following text to French: {text}"
        input_text (str, optional): The input text to be processed by the model. Defaults to None.

    Returns:
        str: The generated text from the model,

In [34]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages3 = [
    SystemMessage("You are expert in giving latest news"),
    HumanMessage("Who is current education minister of India?"), 
]

In [36]:
response = model2.invoke(messages3)

In [37]:
print(response.content)

The current Education Minister of India is **Dharmendra Pradhan**. 

He assumed office on July 8, 2021, and also holds the portfolio of Minister of Skill Development and Entrepreneurship. You can find more information on his official website or the Ministry of Education's website:

*   **Ministry of Education:** [https://education.gov.in/en](https://education.gov.in/en)
*   **Dharmendra Pradhan's Website:** [https://dpradhan.gov.in/](https://dpradhan.gov.in/)






In [38]:
messages4 = [
    SystemMessage("You are expert in giving latest news"),
    HumanMessage("Who is today's date?"), 
]

In [39]:
response4 = model.invoke(messages4)
print(response4.content)

Today is November 21, 2023. 

Do you want to know any news for today? I can provide updates on a variety of topics! Just let me know what you're interested in.



In [40]:
response4.usage_metadata

{'input_tokens': 33,
 'output_tokens': 48,
 'total_tokens': 81,
 'input_token_details': {},
 'output_token_details': {}}

In [ ]:
from datetime import date
from langchain_core.messages import SystemMessage, HumanMessage

system = SystemMessage(
    f"Today's date is {date.today().isoformat()}. "
    "Use this as the current date. Do not rely on your training date."
    "You are expert in coding for python language."
)

messages = [system, HumanMessage("Who is current transport minister of India?")]
print(model.invoke(messages).content)

As of today, July 31, 2026, **Nitin Gadkari** is the current Minister of Road Transport and Highways of India. He also holds the portfolio of Minister of Micro, Small and Medium Enterprises. 

While ministerial positions can change, as of this date, he continues to hold the transport ministry.




